# CSNN Retinal Classification (Small Dataset)
This notebook trains a convolutional spiking neural network (CSNN) on the retinal imagefolder dataset.
It uses helpers in Retinal Classification/CSNN/utils.py and mirrors the structure of the proto k-shot SCNN notebook with lighter defaults.

In [9]:
# Install required packages (run once)
%pip install -q torch torchvision snntorch matplotlib pandas scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.


In [10]:
import sys
import importlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

def resolve_csnn_dir():
    candidates = [
        Path.cwd() / "CSNN",
        Path.cwd() / "Retinal Classification" / "CSNN",
        Path.cwd().parent / "CSNN",
        Path.cwd().parent / "Retinal Classification" / "CSNN",
    ]
    for candidate in candidates:
        if (candidate / "utils.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate CSNN utils.py")

CSNN_DIR = resolve_csnn_dir()
if str(CSNN_DIR) not in sys.path:
    sys.path.insert(0, str(CSNN_DIR))

import utils as csnn_utils
csnn_utils = importlib.reload(csnn_utils)

print("CSNN directory:", CSNN_DIR)
print("utils module:", csnn_utils.__file__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


CSNN directory: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN
utils module: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/utils.py
Torch version: 2.10.0+cu128
CUDA available: True


In [11]:
SEED = 1
IMAGE_SIZE = 64
BATCH_SIZE = 16
DATA_PROP = 0.2
TEST_RATIO = 0.5
NUM_STEPS = 25
EPOCHS = 20
LEARNING_RATE = 1e-3
AUGMENT = True
BALANCE_SAMPLER = True
USE_CLASS_WEIGHTS = True

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = CSNN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)


Device: cuda
Output dir: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs


In [12]:
train_loader, val_loader, class_weights = csnn_utils.build_retinal_dataloaders(
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    data_prop=DATA_PROP,
    num_workers=0,
    seed=SEED,
    augment=AUGMENT,
    balance=BALANCE_SAMPLER,
)

val_dataset = val_loader.dataset
val_targets_full = csnn_utils.get_targets(val_dataset)

def stratified_split_indices(labels, test_ratio=0.5, seed=1):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    val_idx = []
    test_idx = []

    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)

        if len(cls_idx) <= 1:
            n_test = 1
        else:
            n_test = int(round(len(cls_idx) * test_ratio))
            n_test = max(1, n_test)
            n_test = min(n_test, len(cls_idx) - 1)

        test_idx.extend(cls_idx[:n_test].tolist())
        val_idx.extend(cls_idx[n_test:].tolist())

    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return np.array(val_idx), np.array(test_idx)

val_idx, test_idx = stratified_split_indices(val_targets_full, test_ratio=TEST_RATIO, seed=SEED)
val_subset = Subset(val_dataset, val_idx)
test_subset = Subset(val_dataset, test_idx)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

train_targets = csnn_utils.get_targets(train_loader.dataset)
val_targets = csnn_utils.get_targets(val_subset)
test_targets = csnn_utils.get_targets(test_subset)

print("Train class distribution:", csnn_utils.describe_class_distribution(train_targets))
print("Val class distribution:", csnn_utils.describe_class_distribution(val_targets))
print("Test class distribution:", csnn_utils.describe_class_distribution(test_targets))
print("Class weights:", class_weights)
print(f"Split sizes -> train: {len(train_targets)} | val: {len(val_targets)} | test: {len(test_targets)}")


Train class distribution: {0: 289, 1: 59, 2: 160, 3: 31, 4: 47}
Val class distribution: {0: 181, 1: 37, 2: 100, 3: 19, 4: 29}
Test class distribution: {0: 180, 1: 37, 2: 100, 3: 20, 4: 30}
Class weights: tensor([0.4055, 1.9864, 0.7325, 3.7806, 2.4936])
Split sizes -> train: 586 | val: 366 | test: 367


In [13]:
num_classes = len(class_weights)
model = csnn_utils.build_model(
    device=DEVICE,
    input_channels=1,
    input_size=IMAGE_SIZE,
    num_classes=num_classes,
    spike=False,
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE) if USE_CLASS_WEIGHTS else None)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

Sequential(
  (0): Conv2d(1, 12, kernel_size=(5, 5), stride=(1, 1))
  (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (2): Leaky()
  (3): Conv2d(12, 64, kernel_size=(5, 5), stride=(1, 1))
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Leaky()
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=10816, out_features=5, bias=True)
  (8): Leaky()
)


In [14]:
def step_batch(model, data, targets):
    spk_rec, _ = csnn_utils.forward_pass(model, data, num_steps=NUM_STEPS, spike=False)
    spk_count = spk_rec.sum(0)
    loss = loss_fn(spk_count, targets)
    preds = spk_count.argmax(1)
    acc = (preds == targets).float().mean().item()
    return loss, acc, preds, spk_rec

def evaluate_metrics(model, loader, compute_spike_metrics=True):
    model.eval()
    all_preds = []
    all_targets = []
    spike_counts = []
    spike_rates = []

    with torch.no_grad():
        for data, targets in loader:
            data = data.to(DEVICE)
            targets = targets.to(DEVICE)

            _, _, preds, spk_rec = step_batch(model, data, targets)

            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(targets.detach().cpu().numpy())

            if compute_spike_metrics:
                total_spikes = spk_rec.sum(dim=(0, 2))
                spike_counts.append(total_spikes.detach().cpu().numpy())
                spike_rate = total_spikes / float(NUM_STEPS * spk_rec.size(2))
                spike_rates.append(spike_rate.detach().cpu().numpy())

    y_true = np.concatenate(all_targets) if all_targets else np.array([])
    y_pred = np.concatenate(all_preds) if all_preds else np.array([])

    acc = accuracy_score(y_true, y_pred) if y_true.size else 0.0
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    ) if y_true.size else (0.0, 0.0, 0.0, None)

    metrics = {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }

    if compute_spike_metrics:
        if spike_counts:
            spike_counts = np.concatenate(spike_counts)
            spike_rates = np.concatenate(spike_rates)
            metrics.update(
                {
                    "mean_spikes": float(np.mean(spike_counts)),
                    "std_spikes": float(np.std(spike_counts)),
                    "mean_spike_rate": float(np.mean(spike_rates)),
                }
            )
        else:
            metrics.update({"mean_spikes": 0.0, "std_spikes": 0.0, "mean_spike_rate": 0.0})

    return metrics, y_true, y_pred

train_loss_hist = []
val_acc_hist = []

train_start = time.time()
train_start_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start))
print(f"Training start: {train_start_str}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for data, targets in train_loader:
        data = data.to(DEVICE)
        targets = targets.to(DEVICE)

        loss, _, _, _ = step_batch(model, data, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)

    avg_loss = running_loss / len(train_loader.dataset)
    val_metrics, _, _ = evaluate_metrics(model, val_loader, compute_spike_metrics=False)
    val_acc = val_metrics["accuracy"]

    train_loss_hist.append(avg_loss)
    val_acc_hist.append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS} | loss={avg_loss:.4f} | val_acc={val_acc:.4f}")

train_end = time.time()
train_end_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_end))
train_elapsed = train_end - train_start
print(f"Training end: {train_end_str}")
print(f"Training elapsed: {train_elapsed / 60:.2f} min")


Training start: 2026-05-01 16:36:26
Epoch 1/20 | loss=1.6058 | val_acc=0.2705
Epoch 2/20 | loss=1.6698 | val_acc=0.4672
Epoch 3/20 | loss=1.5338 | val_acc=0.3087
Epoch 4/20 | loss=1.5350 | val_acc=0.3169
Epoch 5/20 | loss=1.5869 | val_acc=0.3087
Epoch 6/20 | loss=1.5274 | val_acc=0.3279
Epoch 7/20 | loss=1.4532 | val_acc=0.3880
Epoch 8/20 | loss=1.4498 | val_acc=0.3743
Epoch 9/20 | loss=1.3899 | val_acc=0.2787
Epoch 10/20 | loss=1.3986 | val_acc=0.3634
Epoch 11/20 | loss=1.3446 | val_acc=0.4590
Epoch 12/20 | loss=1.3695 | val_acc=0.2896
Epoch 13/20 | loss=1.3616 | val_acc=0.3224
Epoch 14/20 | loss=1.2996 | val_acc=0.4317
Epoch 15/20 | loss=1.3377 | val_acc=0.4372
Epoch 16/20 | loss=1.2745 | val_acc=0.2951
Epoch 17/20 | loss=1.4005 | val_acc=0.4180
Epoch 18/20 | loss=1.4720 | val_acc=0.4344
Epoch 19/20 | loss=1.3601 | val_acc=0.4262
Epoch 20/20 | loss=1.3166 | val_acc=0.3962
Training end: 2026-05-01 17:00:04
Training elapsed: 23.62 min


In [15]:
history_df = pd.DataFrame({
    "epoch": list(range(1, EPOCHS + 1)),
    "train_loss": train_loss_hist,
    "val_acc": val_acc_hist,
})
history_path = OUTPUT_DIR / "csnn_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved history to:", history_path)

csnn_utils.plot_history(train_loss_hist, val_acc_hist, OUTPUT_DIR)

checkpoint_path = OUTPUT_DIR / "csnn_checkpoint.pth"
torch.save(
    {
        "model_state": model.state_dict(),
        "image_size": IMAGE_SIZE,
        "num_classes": num_classes,
        "num_steps": NUM_STEPS,
        "seed": SEED,
    },
    checkpoint_path,
)
print("Saved checkpoint to:", checkpoint_path)

Saved history to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/csnn_history.csv
Saved checkpoint to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/csnn_checkpoint.pth


In [16]:
if hasattr(train_loader.dataset, "dataset") and hasattr(train_loader.dataset.dataset, "classes"):
    class_names = train_loader.dataset.dataset.classes
else:
    class_names = train_loader.dataset.classes

split_loaders = {
    "train": train_loader,
    "val": val_loader,
    "test": test_loader,
}

metrics_rows = []

for split_name, loader in split_loaders.items():
    metrics, y_true, y_pred = evaluate_metrics(model, loader, compute_spike_metrics=True)
    metrics_row = {"split": split_name}
    metrics_row.update(metrics)
    metrics_rows.append(metrics_row)

    print(
        f"{split_name} -> acc={metrics['accuracy']:.4f}, "
        f"precision={metrics['precision']:.4f}, "
        f"recall={metrics['recall']:.4f}, "
        f"f1={metrics['f1_score']:.4f}"
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = OUTPUT_DIR / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_path = OUTPUT_DIR / f"{split_name}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=f"Confusion Matrix - {split_name}",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    threshold = cm.max() / 2.0 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
            )

    plt.tight_layout()
    cm_img_path = OUTPUT_DIR / f"{split_name}_confusion_matrix.png"
    plt.savefig(cm_img_path, dpi=200)
    plt.close(fig)

    print("Saved report to:", report_path)
    print("Saved confusion matrix to:", cm_path)
    print("Saved confusion matrix image to:", cm_img_path)

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = OUTPUT_DIR / "csnn_split_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved split metrics to:", metrics_path)
metrics_df


train -> acc=0.4522, precision=0.3908, recall=0.4522, f1=0.4009
Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/train_classification_report.csv
Saved confusion matrix to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/train_confusion_matrix.csv
Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/train_confusion_matrix.png
val -> acc=0.3962, precision=0.3274, recall=0.3962, f1=0.3535
Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/val_classification_report.csv
Saved confusion matrix to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/val_confusion_matrix.csv
Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/CSNN/outputs/val_confusion_matrix.png
test -> acc=0.3733, precision=0.3257, recall=0.3733, f1=0.3445
Saved report to: /media/aejaz/New Volume/Projects/SNN

,split,accuracy,precision,recall,f1_score,mean_spikes,std_spikes,mean_spike_rate
0,train,0.452218,0.390772,0.452218,0.400864,2.187713,2.608516,0.017502
1,val,0.396175,0.327402,0.396175,0.353493,1.396175,2.040274,0.011169
2,test,0.373297,0.325692,0.373297,0.344509,1.490463,2.174735,0.011924
